In [ ]:
# ============================================================
# Colab-Ready QNN Architecture (Qiskit) + Visualization
# Hybrid: PCA -> FeatureMap -> Ansatz -> Measurements -> Classifier
# ============================================================

!pip -q install "qiskit>=1.0" qiskit-machine-learning scikit-learn numpy matplotlib qiskit_algorithms pylatexenc

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier

from qiskit_algorithms.optimizers import COBYLA


# -------------------------
# 1) Reproducibility
# -------------------------
np.random.seed(42)

# -------------------------
# 2) Generate high-dimensional dataset
# -------------------------
n_samples  = 600
n_features = 30  # high-dimensional
X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=20,
    n_redundant=5,
    n_classes=2,
    class_sep=1.2,
    flip_y=0.01,
    random_state=42
)

# -------------------------
# 3) Preprocess: scale + PCA -> n_qubits
# -------------------------
n_qubits = 6  # Best practical start (4-8 on simulator)

scaler = StandardScaler()
X = scaler.fit_transform(X)

pca = PCA(n_components=n_qubits, random_state=42)
X_q = pca.fit_transform(X)

# Scale PCA output to [-pi, pi] for stable angle encoding
X_q = np.pi * np.tanh(X_q)

X_train, X_test, y_train, y_test = train_test_split(
    X_q, y, test_size=0.3, random_state=42, stratify=y
)

# -------------------------
# 4) Build QNN circuit: FeatureMap + Ansatz
# -------------------------
feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2, entanglement="linear")
ansatz      = RealAmplitudes(num_qubits=n_qubits, reps=2, entanglement="linear")

qc = feature_map.compose(ansatz)

# Visualize the circuit
print("QNN Circuit (FeatureMap + Ansatz):")
display(qc.draw("mpl"))

# -------------------------
# 5) Define EstimatorQNN
#    - Inputs: feature_map parameters
#    - Weights: ansatz parameters
# -------------------------
estimator = StatevectorEstimator()

input_params  = list(feature_map.parameters)
weight_params = list(ansatz.parameters)

# By default, EstimatorQNN measures expectation of Z...Z observable
# That works well for binary classification with a classical loss.
qnn = EstimatorQNN(
    circuit=qc,
    input_params=input_params,
    weight_params=weight_params,
    estimator=estimator
)

# -------------------------
# 6) Train classifier
# -------------------------
optimizer = COBYLA(maxiter=120)

clf = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

clf.fit(X_train, y_train)

# -------------------------
# 7) Evaluate
# -------------------------
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\nAccuracy (QNN) = {acc:.4f}")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("QNN Confusion Matrix")
plt.show()